In [ ]:
pip install -U transformers

In [ ]:
import transformers
print(transformers.__version__)


5.3.0


In [ ]:
pip install datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("go_emotions")
print(dataset)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})


In [ ]:
print(dataset["train"].features["labels"].feature.names)


['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [ ]:
positive = {
    'admiration','amusement','approval','caring','desire',
    'excitement','gratitude','joy','love','optimism',
    'pride','relief'
}


In [ ]:
negative = {
    'anger','annoyance','disappointment','disapproval',
    'disgust','embarrassment','fear','grief',
    'nervousness','remorse','sadness'
}


In [ ]:
 neutral = {'neutral'}


In [ ]:
label_names = dataset["train"].features["labels"].feature.names

def map_to_sentiment(example):
    emotions = [label_names[i] for i in example["labels"]]

    has_pos = any(e in positive for e in emotions)
    has_neg = any(e in negative for e in emotions)

    # Remove mixed sentiment
    if has_pos and has_neg:
        example["sentiment"] = -1   # mark for removal
    elif has_pos:
        example["sentiment"] = 2   # positive
    elif has_neg:
        example["sentiment"] = 0   # negative
    else:
        example["sentiment"] = 1   # neutral

    return example

dataset = dataset.map(map_to_sentiment)


In [ ]:
dataset = dataset.filter(lambda x: x["sentiment"] != -1)


In [ ]:
dataset = dataset.remove_columns(["labels", "id"])


In [ ]:
from collections import Counter

Counter(dataset["train"]["sentiment"])


Counter({1: 16983, 0: 9017, 2: 16588})

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = dataset["train"]["sentiment"]

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = np.array(class_weights, dtype=np.float32)

print(class_weights)


[1.5743595 0.8358947 0.8557994]


In [ ]:
dataset = dataset.rename_column("sentiment", "labels")


In [ ]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(tokenize, batched=True)


Map:   0%|          | 0/42588 [00:00<?, ? examples/s]

Map:   0%|          | 0/5303 [00:00<?, ? examples/s]

Map:   0%|          | 0/5349 [00:00<?, ? examples/s]

In [ ]:
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)


In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
import torch
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):  # ✅ fix here
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fn = torch.nn.CrossEntropyLoss(
            weight=torch.tensor(class_weights).to(model.device)
        )

        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss



In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
)


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.617597,0.612873
2,0.472684,0.641347
3,0.324649,1.004974


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=15972, training_loss=0.4899241932048076, metrics={'train_runtime': 1832.1504, 'train_samples_per_second': 69.734, 'train_steps_per_second': 8.718, 'total_flos': 4231216636867584.0, 'train_loss': 0.4899241932048076, 'epoch': 3.0})

In [ ]:
# Save model and tokenizer
model.save_pretrained("sentiment_model")
tokenizer.save_pretrained("sentiment_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('sentiment_model/tokenizer_config.json', 'sentiment_model/tokenizer.json')

In [ ]:
from google.colab import files
import shutil

shutil.make_archive("sentiment_model", 'zip', "sentiment_model")
files.download("sentiment_model.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from transformers import pipeline

# 1️⃣ Load your trained model and tokenizer
classifier = pipeline(
    "text-classification",
    model="sentiment_model",      # path to your trained model
    tokenizer="sentiment_model"
)

# 2️⃣ Map Hugging Face labels to actual sentiment
label_map = {
    "LABEL_0": "Negative",
    "LABEL_1": "Neutral",
    "LABEL_2": "Positive"
}

# 3️⃣ Define prediction function with uncertainty handling
def predict_sentiment(text, threshold=0.70):
    result = classifier(text)
    score = result[0]['score']
    label = result[0]['label']

    # If the model is not confident, classify as Neutral
    if score < threshold:
        sentiment = "Neutral"
    else:
        sentiment = label_map[label]

    return sentiment, score

# 4️⃣ Test examples
texts = [
    "I love the taste but hate its packaging",
    "The product is worst, low quality",
    "I hate the flavor, it is too sour"
]

for t in texts:
    sentiment, conf = predict_sentiment(t)
    print(f"Text: {t}")
    print(f"Predicted Sentiment: {sentiment} (Confidence: {conf:.2f})\n")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: I love the taste but hate its packaging
Predicted Sentiment: Positive (Confidence: 0.93)

Text: The product is worst, low quality
Predicted Sentiment: Negative (Confidence: 0.96)

Text: I hate the flavor, it is too sour
Predicted Sentiment: Negative (Confidence: 0.98)

